# Regresión Lineal Regularizada: Ridge y Elastic Net
**Proyecto P1 – Optimización, UTEC 2026** · Segunda entrega: validación empírica

Dataset: precios de alquiler de inmuebles en Lima (Properati Perú, oct-2022 a mar-2023).

Este notebook reúne todo el pipeline del informe (`01_clean.py` … `05_gd_check.py` + `models.py`) en un solo flujo:

1. Limpieza del dataset crudo
2. Matriz de diseño (p = 34)
3. Implementaciones *from scratch* (Ridge y Elastic Net)
4. Partición train/test y estandarización
5. Validación cruzada (Ridge y Elastic Net)
6. Ajuste final y métricas
7. Validación contra scikit-learn
8. Camino de regularización
9. Descenso del gradiente para Ridge (convergencia y número de condición)
10. Interpretación de coeficientes

> **Requisito:** colocar `dataset_raw.csv` en la carpeta indicada en `DATA_DIR`.

In [ ]:
import re, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, KFold
from sklearn.linear_model import Ridge, ElasticNet

np.random.seed(42)

DATA_DIR = '../data'          # <- ajusta si tu CSV está en otra carpeta
RAW = f'{DATA_DIR}/dataset_raw.csv'
TASA_USD_PEN = 3.80           # tipo de cambio de referencia (feb-mar 2023)

ACENTO, GRIS = '#005F69', '#6E6E6E'
plt.rcParams.update({'font.size': 10, 'axes.spines.top': False,
                     'axes.spines.right': False, 'figure.dpi': 110})

## 1. Limpieza de datos
El texto semiestructurado ("S/.2,600", "3 dormitorios", "70 m²") se convierte a variables numéricas.

In [ ]:
df = pd.read_csv(RAW)
print("Filas originales:", len(df))

def parse_price(s):
    s = str(s).strip()
    is_usd = s.startswith('USD')
    digits = re.sub(r'[^\d]', '', s.replace(',', ''))
    if digits == '':
        return np.nan
    val = float(digits)
    return val * TASA_USD_PEN if is_usd else val

def parse_count(s):
    if pd.isna(s): return np.nan
    m = re.match(r'(\d+)', str(s))
    return float(m.group(1)) if m else np.nan

def parse_area(s):
    if pd.isna(s): return np.nan
    m = re.match(r'([\d.,]+)\s*m', str(s))
    return float(m.group(1).replace(',', '')) if m else np.nan

def parse_year(s):
    if pd.isna(s): return np.nan
    s = str(s)
    if not re.match(r'^\d{4}$', s): return np.nan   # filas con 'NN m²' por error de captura
    y = int(s)
    return y if 1900 <= y <= 2023 else np.nan

def parse_maint(s):
    if pd.isna(s): return np.nan
    digits = re.sub(r'[^\d]', '', str(s).replace(',', ''))
    return float(digits) if digits else np.nan

def parse_location(s):
    # "[Sub-zona,] Distrito, Provincia, Departamento"
    parts = [p.strip() for p in str(s).split(',') if p.strip()]
    if len(parts) >= 3:   return pd.Series([parts[-3], parts[-1]])
    elif len(parts) == 2: return pd.Series([parts[0], parts[-1]])
    return pd.Series([np.nan, np.nan])

df['price_pen']   = df['price'].apply(parse_price)
df['bedrooms']    = df['bedroom'].apply(parse_count)
df['bathrooms']   = df['bathroom'].apply(parse_count)
df['area_m2']     = df['area'].apply(parse_area)
df['year_built']  = df['year_contruction'].apply(parse_year)
df['age_years']   = 2023 - df['year_built']
df['maintenance_pen'] = df['maintenance'].apply(parse_maint)
df['has_maintenance'] = df['maintenance_pen'].notna().astype(int)
df['maintenance_pen'] = df['maintenance_pen'].fillna(0.0)
df[['district', 'department']] = df['location'].apply(parse_location)
df['district'] = df['district'].str.strip()
df['department'] = df['department'].str.strip()

In [ ]:
# Filtros de calidad
df = df[df['housing_type'].isin(['Apartamento', 'Casa'])]
df = df[df['department'] == 'Lima']
df = df.dropna(subset=['price_pen', 'bedrooms', 'bathrooms', 'area_m2'])
df = df[(df['price_pen'].between(300, 30000)) & (df['area_m2'].between(15, 1000))]
df = df[(df['bedrooms'] <= 10) & (df['bathrooms'] <= 10)]
print("Filas tras limpieza:", len(df))
print(df['district'].value_counts().head(15))

# Distritos con >= 15 anuncios; el resto -> 'Otro'
top_districts = df['district'].value_counts()
top_districts = top_districts[top_districts >= 15].index.tolist()
df['district_grp'] = df['district'].where(df['district'].isin(top_districts), 'Otro')
print("\nDistritos usados como dummy:", top_districts)

df_clean = df[['price_pen', 'bedrooms', 'bathrooms', 'area_m2', 'age_years',
               'maintenance_pen', 'has_maintenance', 'housing_type', 'district_grp']].copy()
df_clean['age_years'] = df_clean['age_years'].fillna(df_clean['age_years'].median())
df_clean = df_clean.reset_index(drop=True)
df_clean.describe()

In [ ]:
# Asimetría del precio vs. log-precio (justifica la transformación log)
from scipy.stats import skew
print(f"skew(precio)     = {skew(df_clean['price_pen']):.2f}")
print(f"skew(log precio) = {skew(np.log(df_clean['price_pen'])):.2f}")

fig, axes = plt.subplots(1, 2, figsize=(9, 3.2))
axes[0].hist(df_clean['price_pen'], bins=40, color=ACENTO); axes[0].set_title('Precio (S/.)')
axes[1].hist(np.log(df_clean['price_pen']), bins=40, color=ACENTO); axes[1].set_title('log(precio)')
plt.tight_layout(); plt.show()

## 2. Matriz de diseño
Variables base + dummies (referencia: tipo = Apartamento, distrito = primera categoría alfabética) + interacciones + términos cuadráticos → **p = 34**.

In [ ]:
y = np.log(df_clean['price_pen'].values)

num_cols = ['bedrooms', 'bathrooms', 'area_m2', 'age_years', 'maintenance_pen']
X_num = df_clean[num_cols].copy()

house_dum = pd.get_dummies(df_clean['housing_type'], prefix='tipo', drop_first=True, dtype=float)
dist_dum  = pd.get_dummies(df_clean['district_grp'], prefix='distrito', drop_first=True, dtype=float)
has_maint = df_clean[['has_maintenance']].astype(float)

inter = pd.DataFrame({
    'area*bedrooms':      X_num['area_m2'] * X_num['bedrooms'],
    'area*bathrooms':     X_num['area_m2'] * X_num['bathrooms'],
    'bedrooms*bathrooms': X_num['bedrooms'] * X_num['bathrooms'],
})
for c in dist_dum.columns:
    inter[f'area*{c}'] = X_num['area_m2'] * dist_dum[c]

quad = pd.DataFrame({'area_m2^2': X_num['area_m2'] ** 2,
                     'age_years^2': X_num['age_years'] ** 2})

Xfull = pd.concat([X_num, has_maint, house_dum, dist_dum, inter, quad], axis=1).astype(float)
feat_names = Xfull.columns.tolist()
X = Xfull.values
n, p = X.shape
print(f"n = {n}   p = {p}")
print(feat_names)

In [ ]:
# Multicolinealidad entre dormitorios, baños y área
Xfull[['bedrooms', 'bathrooms', 'area_m2']].corr().round(2)

## 3. Implementaciones *from scratch*
- **Ridge**: solución cerrada $\hat\beta=(X^\top X+\lambda I)^{-1}X^\top y$
- **Elastic Net**: descenso por coordenadas con umbralización suave

$$\beta_j \leftarrow \frac{S_{\lambda\alpha}(x_j^\top r_j)}{\|x_j\|^2+\lambda(1-\alpha)}$$

Convención: función objetivo **sin** el factor $1/n$ (ver equivalencia con scikit-learn en la sección 7).

In [ ]:
def soft_threshold(z, tau):
    return np.sign(z) * np.maximum(np.abs(z) - tau, 0.0)


class RidgeScratch:
    """Ridge con solución cerrada."""
    def __init__(self, lam, fit_intercept=True):
        self.lam, self.fit_intercept = lam, fit_intercept

    def fit(self, X, y):
        n, p = X.shape
        Xc = X - X.mean(axis=0) if self.fit_intercept else X
        yc = y - y.mean() if self.fit_intercept else y
        self.coef_ = np.linalg.solve(Xc.T @ Xc + self.lam * np.eye(p), Xc.T @ yc)
        self.intercept_ = (y.mean() - X.mean(axis=0) @ self.coef_) if self.fit_intercept else 0.0
        return self

    def predict(self, X):
        return self.intercept_ + X @ self.coef_


class ElasticNetScratch:
    """Elastic Net por descenso por coordenadas (Algoritmo 1 del informe)."""
    def __init__(self, lam, alpha, tol=1e-6, max_iter=2000, fit_intercept=True):
        self.lam, self.alpha, self.tol = lam, alpha, tol
        self.max_iter, self.fit_intercept = max_iter, fit_intercept

    def fit(self, X, y, warm_start=None, track_loss=False):
        n, p = X.shape
        Xc = X - X.mean(axis=0) if self.fit_intercept else X
        yc = y - y.mean() if self.fit_intercept else y
        xj_norm2 = np.sum(Xc ** 2, axis=0)

        beta = np.zeros(p) if warm_start is None else warm_start.copy()
        r = yc - Xc @ beta
        loss_path = [] if track_loss else None
        lam, alpha = self.lam, self.alpha

        for it in range(1, self.max_iter + 1):
            max_change = 0.0
            for j in range(p):
                if xj_norm2[j] == 0:
                    continue
                r_j = r + Xc[:, j] * beta[j]            # residuo parcial
                z_j = Xc[:, j] @ r_j
                beta_new = soft_threshold(z_j, lam * alpha) / (xj_norm2[j] + lam * (1 - alpha))
                delta = beta_new - beta[j]
                if delta != 0.0:
                    r = r_j - Xc[:, j] * beta_new
                    beta[j] = beta_new
                    max_change = max(max_change, abs(delta))
            if track_loss:
                loss_path.append(0.5 * np.sum(r ** 2) + lam * (alpha * np.sum(np.abs(beta))
                                 + 0.5 * (1 - alpha) * np.sum(beta ** 2)))
            if max_change < self.tol:
                break

        self.coef_ = beta
        self.intercept_ = (y.mean() - X.mean(axis=0) @ beta) if self.fit_intercept else 0.0
        self.n_iter_, self.loss_path_ = it, loss_path
        return self

    def predict(self, X):
        return self.intercept_ + X @ self.coef_

### Verificación con el ejemplo numérico del informe
$X=\begin{pmatrix}1&1\\1&0\\0&1\end{pmatrix}$, $y=(3,1,2)^\top$, sin intercepto.  
Esperado: Ridge ($\lambda=1$) → (0.875, 1.375); Elastic Net ($\lambda=1,\alpha=0.5$) → (0.8095, 1.4762).

In [ ]:
Xe = np.array([[1, 1], [1, 0], [0, 1]], float); ye = np.array([3, 1, 2], float)
print("Ridge  :", RidgeScratch(lam=1.0, fit_intercept=False).fit(Xe, ye).coef_)
print("EN     :", ElasticNetScratch(lam=1.0, alpha=0.5, tol=1e-12, max_iter=10000,
                                    fit_intercept=False).fit(Xe, ye).coef_)
print("EN lam=8:", ElasticNetScratch(lam=8.0, alpha=0.5, tol=1e-12, max_iter=10000,
                                     fit_intercept=False).fit(Xe, ye).coef_, "(esperado (0, 1/6))")

## 4. Partición train/test y estandarización
Estandarización **solo con estadísticas de train** (evita fuga de información).

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

mu, sigma = X_train.mean(axis=0), X_train.std(axis=0)
sigma[sigma == 0] = 1.0
X_train_s = (X_train - mu) / sigma
X_test_s  = (X_test  - mu) / sigma
print("n_train =", len(y_train), " n_test =", len(y_test))

def metrics(y_true, y_pred):
    resid = y_true - y_pred
    mse = np.mean(resid ** 2)
    ss_res, ss_tot = np.sum(resid ** 2), np.sum((y_true - y_true.mean()) ** 2)
    return dict(R2=1 - ss_res / ss_tot, RMSE=np.sqrt(mse), MSE=mse, MAE=np.mean(np.abs(resid)))

## 5. Validación cruzada (5 pliegues)
### 5.1 Ridge — grilla de λ (escala log) y regla 1-SE

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

def cv_mse(make_model, Xs, ys):
    errs = []
    for tr, va in kf.split(Xs):
        Xtr, Xva = Xs[tr], Xs[va]
        m2, s2 = Xtr.mean(0), Xtr.std(0); s2[s2 == 0] = 1.0
        model = make_model().fit((Xtr - m2) / s2, ys[tr])
        errs.append(np.mean((ys[va] - model.predict((Xva - m2) / s2)) ** 2))
    return np.mean(errs), np.std(errs)

lambdas_ridge = np.logspace(-2, 3, 25)
rows = [(lam, *cv_mse(lambda: RidgeScratch(lam=lam), X_train_s, y_train)) for lam in lambdas_ridge]
cv_ridge = pd.DataFrame(rows, columns=['lambda', 'cv_mse', 'cv_std'])

i = cv_ridge['cv_mse'].idxmin()
lam_min, mse_min = cv_ridge.loc[i, 'lambda'], cv_ridge.loc[i, 'cv_mse']
se_min = cv_ridge.loc[i, 'cv_std'] / np.sqrt(5)
lam_1se = cv_ridge[cv_ridge['cv_mse'] <= mse_min + se_min]['lambda'].max()
print(f"Ridge: lambda_min = {lam_min:.3f} (MSE cv = {mse_min:.4f})   lambda_1se = {lam_1se:.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(5.6, 3.8))
se = cv_ridge['cv_std'] / np.sqrt(5)
ax.plot(cv_ridge['lambda'], cv_ridge['cv_mse'], color=ACENTO, lw=1.6)
ax.fill_between(cv_ridge['lambda'], cv_ridge['cv_mse'] - se, cv_ridge['cv_mse'] + se,
                color=ACENTO, alpha=0.18, lw=0)
ax.axvline(lam_min, color='crimson', ls=':', label=rf"$\lambda_{{\min}}={lam_min:.2f}$")
ax.axvline(lam_1se, color=GRIS, ls=':', label=rf"$\lambda_{{1se}}={lam_1se:.2f}$")
ax.set_xscale('log'); ax.set_xlabel(r'$\lambda$ (escala log)')
ax.set_ylabel('MSE promedio (CV, 5 pliegues)'); ax.legend(frameon=False)
plt.tight_layout(); plt.show()

### 5.2 Elastic Net — grilla bidimensional (λ, α)  *(puede tardar unos minutos)*

In [ ]:
lambdas_en = np.logspace(-1.5, 2, 14)
alphas_en = [0.1, 0.3, 0.5, 0.7, 0.9, 1.0]

rows = []
for alpha in alphas_en:
    for lam in lambdas_en:
        m, s = cv_mse(lambda: ElasticNetScratch(lam=lam, alpha=alpha, tol=1e-5, max_iter=3000),
                      X_train_s, y_train)
        rows.append((alpha, lam, m, s))
cv_en = pd.DataFrame(rows, columns=['alpha', 'lambda', 'cv_mse', 'cv_std'])

j = cv_en['cv_mse'].idxmin()
alpha_opt, lam_opt = cv_en.loc[j, 'alpha'], cv_en.loc[j, 'lambda']
print(f"Elastic Net óptimo: alpha = {alpha_opt}, lambda = {lam_opt:.3f}, MSE cv = {cv_en.loc[j,'cv_mse']:.4f}")
cv_en.pivot(index='lambda', columns='alpha', values='cv_mse').round(4)

## 6. Ajuste final y desempeño en test

In [ ]:
ridge_final = RidgeScratch(lam=lam_min).fit(X_train_s, y_train)
en_final = ElasticNetScratch(lam=lam_opt, alpha=alpha_opt, tol=1e-8, max_iter=8000).fit(
    X_train_s, y_train, track_loss=True)
ols = RidgeScratch(lam=1e-8).fit(X_train_s, y_train)       # MCO de referencia

pred_te_r, pred_te_e = ridge_final.predict(X_test_s), en_final.predict(X_test_s)

tabla = pd.DataFrame({
    'MCO (sin regularizar)': metrics(y_test, ols.predict(X_test_s)),
    f'Ridge (λ={lam_min:.2f})': metrics(y_test, pred_te_r),
    f'Elastic Net (λ={lam_opt:.2f}, α={alpha_opt})': metrics(y_test, pred_te_e),
}).T
tabla['Error típico (%)'] = (np.exp(tabla['MAE']) - 1) * 100   # exp(MAE) - 1
display(tabla.round(4))

n_active_en = int(np.sum(np.abs(en_final.coef_) > 1e-8))
print(f"\nEN: {en_final.n_iter_} ciclos, {n_active_en}/{p} variables activas")
print("Variables eliminadas:", [f for f, c in zip(feat_names, en_final.coef_) if abs(c) <= 1e-8])

In [ ]:
# Predicho vs real
fig, axes = plt.subplots(1, 2, figsize=(8.6, 4.0), sharex=True, sharey=True)
for ax, pred, name in zip(axes, [pred_te_r, pred_te_e], ['Ridge', 'Elastic Net']):
    ax.scatter(y_test, pred, s=14, alpha=0.55, color=ACENTO, edgecolors='none')
    lims = [min(y_test.min(), pred.min()) - 0.1, max(y_test.max(), pred.max()) + 0.1]
    ax.plot(lims, lims, '--', color=GRIS, lw=1); ax.set_xlim(lims); ax.set_ylim(lims)
    ax.set_xlabel('log(precio) real'); ax.set_title(f"{name}  ($R^2={metrics(y_test, pred)['R2']:.3f}$)")
axes[0].set_ylabel('log(precio) predicho')
plt.tight_layout(); plt.show()

# Residuos
resid = y_test - pred_te_e
fig, ax = plt.subplots(figsize=(5.6, 3.8))
ax.scatter(pred_te_e, resid, s=14, alpha=0.55, color=ACENTO, edgecolors='none')
ax.axhline(0, color=GRIS, ls='--', lw=1)
ax.set_xlabel('log(precio) predicho (Elastic Net)'); ax.set_ylabel('Residuo')
plt.tight_layout(); plt.show()

## 7. Validación contra scikit-learn
Equivalencias (Tabla 2 del informe):
- Ridge: `alpha_sk = λ`
- Elastic Net: `alpha_sk = λ / n`, `l1_ratio = α`

In [ ]:
sk_ridge = Ridge(alpha=lam_min).fit(X_train_s, y_train)
sk_en = ElasticNet(alpha=lam_opt / len(y_train), l1_ratio=alpha_opt,
                   tol=1e-10, max_iter=200000).fit(X_train_s, y_train)

print("Ridge  max|Δcoef| =", np.max(np.abs(ridge_final.coef_ - sk_ridge.coef_)),
      "  Δintercepto =", abs(ridge_final.intercept_ - sk_ridge.intercept_))
print("EN     max|Δcoef| =", np.max(np.abs(en_final.coef_ - sk_en.coef_)),
      "  Δintercepto =", abs(en_final.intercept_ - sk_en.intercept_))
print("R² test  Ridge sk:", round(metrics(y_test, sk_ridge.predict(X_test_s))['R2'], 7),
      "| EN sk:", round(metrics(y_test, sk_en.predict(X_test_s))['R2'], 7))

## 8. Camino de regularización (Elastic Net, α = α⋆ fijo)
Con *warm start* sobre una grilla decreciente de λ.

In [ ]:
lam_path = np.logspace(np.log10(lambdas_en.max()) + 0.5, -1.8, 60)
coef_path, n_active_path, warm = [], [], np.zeros(p)
for lam in lam_path:
    m = ElasticNetScratch(lam=lam, alpha=alpha_opt, tol=1e-6, max_iter=3000)
    m.fit(X_train_s, y_train, warm_start=warm)
    warm = m.coef_.copy()
    coef_path.append(warm.copy()); n_active_path.append(int(np.sum(np.abs(warm) > 1e-8)))
coef_path = np.array(coef_path)

fig, ax = plt.subplots(figsize=(6.6, 4.4))
cmap = plt.get_cmap('tab20')
top_idx = np.argsort(-np.abs(coef_path[-1]))[:10]
for k, jj in enumerate(top_idx):
    ax.plot(lam_path, coef_path[:, jj], color=cmap(k % 20), lw=1.3, label=feat_names[jj])
for jj in range(p):
    if jj not in top_idx:
        ax.plot(lam_path, coef_path[:, jj], color='0.85', lw=0.7, zorder=0)
ax.axvline(lam_opt, color='crimson', ls=':')
ax.set_xscale('log'); ax.invert_xaxis()
ax.set_xlabel(r'$\lambda$ (escala log, decreciente)'); ax.set_ylabel(r'$\hat\beta_j$ (estandarizados)')
ax.legend(fontsize=6.5, ncol=2, frameon=False, loc='upper left')
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(figsize=(5.6, 3.4))
ax.plot(lam_path, n_active_path, color=ACENTO, lw=1.6, marker='o', ms=2.5)
ax.axvline(lam_opt, color='crimson', ls=':', label=rf"$\lambda^\star={lam_opt:.2f}$")
ax.set_xscale('log'); ax.invert_xaxis()
ax.set_xlabel(r'$\lambda$ (escala log, decreciente)'); ax.set_ylabel('N.º de coeficientes no nulos')
ax.legend(frameon=False); plt.tight_layout(); plt.show()

## 9. Convergencia del descenso por coordenadas
$F_E(\beta_k)-F_E^\star$ en escala logarítmica.

In [ ]:
loss = np.array(en_final.loss_path_)
fig, ax = plt.subplots(figsize=(5.6, 3.8))
ax.plot(np.arange(1, len(loss) + 1), loss - loss.min() + 1e-10, color=ACENTO, lw=1.5)
ax.set_yscale('log'); ax.set_xlabel('Ciclo del descenso por coordenadas')
ax.set_ylabel(r'$F_E(\beta_k)-F_E^\star$')
plt.tight_layout(); plt.show()

## 10. Ridge por descenso del gradiente
Con $\eta = 1/L$, $L=s_1^2+\lambda$, $\mu=s_p^2+\lambda$, $\kappa=L/\mu$. La Proposición 3.2 da la cota
$\|\beta_k-\hat\beta_R\|\le(1-1/\kappa)^k\|\beta_0-\hat\beta_R\|$.

In [ ]:
Xc = X_train_s - X_train_s.mean(axis=0)
yc = y_train - y_train.mean()
eig = np.linalg.eigvalsh(Xc.T @ Xc)
L, mu_cvx = eig.max() + lam_min, eig.min() + lam_min
kappa, eta = L / mu_cvx, 1.0 / L
print(f"L = {L:.1f}   mu = {mu_cvx:.3f}   kappa = {kappa:.0f}   eta = {eta:.2e}")

beta, tol, max_iter = np.zeros(p), 1e-10, 200000
history = []
for k in range(1, max_iter + 1):
    grad = Xc.T @ (Xc @ beta - yc) + lam_min * beta
    beta_new = beta - eta * grad
    diff = np.linalg.norm(beta_new - beta)
    beta = beta_new
    if k % 20 == 0 or k == 1:
        history.append((k, np.linalg.norm(beta - ridge_final.coef_)))
    if diff < tol:
        break

print("Iteraciones hasta convergencia:", k)
print("max|β_GD − β_cerrada| =", np.max(np.abs(beta - ridge_final.coef_)))
print("max|β_GD − sklearn|   =", np.max(np.abs(beta - sk_ridge.coef_)))
print("(1−1/κ)^k             =", (1 - 1 / kappa) ** k)

In [ ]:
hist = np.array(history)
fig, ax = plt.subplots(figsize=(5.6, 3.8))
ax.plot(hist[:, 0], hist[:, 1], color=ACENTO, lw=1.5)
ax.set_yscale('log'); ax.set_xlabel('Iteración'); ax.set_ylabel(r'$\|\beta_k-\hat\beta_R\|$')
plt.tight_layout(); plt.show()

## 11. Interpretación de coeficientes
En el modelo log-lineal, con variable estandarizada, el efecto porcentual por unidad original es $\exp(\hat\beta_j/\sigma_j)-1$.

In [ ]:
order = np.argsort(-np.abs(en_final.coef_))[:12]
top = pd.DataFrame({
    'variable': [feat_names[i] for i in order],
    'coef_EN (est.)': en_final.coef_[order],
    'coef_Ridge (est.)': ridge_final.coef_[order],
    'efecto % por unidad': [(np.exp(en_final.coef_[i] / sigma[i]) - 1) * 100 for i in order],
})
display(top.round(4))

# Ridge vs Elastic Net
fig, ax = plt.subplots(figsize=(6.6, 4.6))
yp, h = np.arange(len(order)), 0.35
ax.barh(yp + h/2, ridge_final.coef_[order], height=h, color=GRIS, label='Ridge')
ax.barh(yp - h/2, en_final.coef_[order], height=h, color=ACENTO, label='Elastic Net')
ax.set_yticks(yp); ax.set_yticklabels(top['variable'], fontsize=8); ax.invert_yaxis()
ax.axvline(0, color='k', lw=0.8); ax.set_xlabel(r'Coeficiente estandarizado $\hat\beta_j$')
ax.legend(frameon=False); plt.tight_layout(); plt.show()

### Nota sobre signos "contraintuitivos"
Distritos como La Molina o Santiago de Surco pueden tener coeficiente negativo pese a su precio medio alto: el coeficiente es un efecto **condicional** al área (y demás variables). Estos distritos tienen áreas grandes pero menor precio por m² que Barranco/San Isidro/Miraflores, así que una vez controlado el tamaño, el ajuste por distrito resulta negativo.

In [ ]:
# Precio por m² y área media por distrito (apoya la nota anterior)
g = df_clean.assign(pm2=df_clean['price_pen'] / df_clean['area_m2']).groupby('district_grp')
g.agg(n=('price_pen', 'size'), area_media=('area_m2', 'mean'),
      precio_medio=('price_pen', 'mean'), precio_m2_mediano=('pm2', 'median')).round(1).sort_values('precio_m2_mediano', ascending=False)

## 12. Limitaciones
- Tipo de cambio fijo (3.80), no el vigente por anuncio.
- Son precios de **oferta**, no de cierre.
- Corte temporal corto (oct-2022 a mar-2023): sin control de estacionalidad.
- Filas descartadas por errores de captura (año de construcción) pueden introducir sesgo de selección.